# ZnO–CO2 analysis of a simulation run

Analyses of one run in `results/simulations/<RUN_ID>/`, finished or still running. Each
section runs the analysis workflow exactly as the launcher does:

```bash
python -m scripts.workflows.run_analysis <method> --config configs/analysis/zno_current.yaml \
    --scientific-results results/simulations/<RUN_ID> --run-id <OUTPUT_ID>_<section>
```

(`./run_analysis.sh <method> ...` is the same command inside the analysis environment.)
The YAML sets every option (CO2RR variant, NEB resolution, MD threshold, figure seed and
dpi); the notebook only selects the run and shows the files. Each section writes a new
analysis run `results/analyses/<OUTPUT_ID>_<section>/` with its `run_manifest.json`
and logs; an existing ID is refused, so change `OUTPUT_ID` to rerun a section. The
simulation run is read only, except `md/<material>/<site>/md_frame_analysis.csv`, which
the MD analysis writes there (the dataset builder reads it).

Run the cells from `notebooks/` or the repository root. Nothing runs until a `RUN_*`
flag in the configuration cell is set to `True`.

| Section | Method | Writes into `results/analyses/<OUTPUT_ID>_<section>/` |
| --- | --- | --- |
| 2 CO2RR | `co2_reduction` (+ `co2rr_corrections`) | `science/co2rr/`, `science/co2_reduction/` |
| 3 NEB | `neb` | `science/neb/` |
| 4 MD | `md_trajectory` | `science/md/` |
| 5 Paper | `paper` | `paper/` and `science/` |

## Local execution

Interactive: open Jupyter in `notebooks/` with an environment that has ASE, pandas,
matplotlib and PyYAML (`qml-zno-simulation`, or `gpaw-dft` on the lab PC), set the flags
and run all cells. Headless:

```bash
cd notebooks
QML_ZNO_SIM_ENV=gpaw-dft ./run_zno_notebook.sh zno-co2-analysis.ipynb zno-co2-analysis.executed.ipynb
```

In [ ]:
from pathlib import Path
import subprocess
import sys
import os
import csv

CWD = Path.cwd()
NOTEBOOK_DIR = next(
    (path for path in [CWD, CWD / "notebooks", CWD.parent / "notebooks"] if (path / "zno-co2-analysis.ipynb").exists()),
    CWD,
)
sim_candidates = [
    CWD,
    CWD.parent,
    NOTEBOOK_DIR.parent,
]
LUCAS_DIR = next((path for path in sim_candidates if (path / "scripts" / "workflows" / "run_simulations.py").exists()), None)
if LUCAS_DIR is None:
    raise FileNotFoundError(
        "Could not resolve simulation scripts. Run from notebooks/ or the repository root."
    )

print(f"Current directory: {CWD}")
print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Simulator directory: {LUCAS_DIR}")

def run_cmd(args: list[str], env: dict | None = None) -> None:
    """Run a command from the repository root and stream its output into the cell."""
    print("$", " ".join(args), flush=True)
    with subprocess.Popen(args, cwd=LUCAS_DIR, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
    if process.returncode:
        raise subprocess.CalledProcessError(process.returncode, args)

def show_table(path: Path, columns: list[str] | None = None, **filters) -> None:
    """Print a CSV written by a module; rows can be filtered by column=value."""
    if not path.exists():
        print(f"[missing] {path}")
        return
    with path.open("r", newline="", encoding="utf-8") as handle:
        rows = [row for row in csv.DictReader(handle)
                if all(row.get(key) == value for key, value in filters.items())]
    print(f"== {path} ({len(rows)} rows)")
    if not rows:
        return
    keep = [name for name in (columns or list(rows[0])) if name in rows[0]]
    try:
        import pandas as pd
        with pd.option_context("display.width", 200, "display.max_columns", None, "display.max_rows", 200):
            print(pd.DataFrame(rows)[keep].to_string(index=False))
    except ImportError:
        print(" | ".join(keep))
        for row in rows:
            print(" | ".join(str(row.get(name, "")) for name in keep))

def list_files(directory: Path, patterns=("*.png", "*.pdf", "*.svg", "*.csv", "*.md")) -> None:
    """List the files a module wrote (figures and tables)."""
    if not directory.is_dir():
        print(f"[missing] {directory}")
        return
    found = sorted({path for pattern in patterns for path in directory.rglob(pattern)})
    for path in found:
        print(f"[OK] {path}")
    if not found:
        print(f"[empty] {directory}")

## Run configuration

Edit only this cell. The simulation settings were fixed by the run (`config.yaml` in the
run directory); the analysis settings come from `ANALYSIS_CONFIG`.

In [ ]:
RUN_ID = "zno_ic_final"                          # results/simulations/<RUN_ID>/
OUTPUT_ID = f"{RUN_ID}_notebook"                 # results/analyses/<OUTPUT_ID>_<section>/
ANALYSIS_CONFIG = "configs/analysis/zno_current.yaml"
MATERIAL = "ZnO"                                 # input.material of ANALYSIS_CONFIG (to show files)

RUN_STATUS = False   # Read-only status dashboard of the run
RUN_CO2RR  = False   # CO2RR corrections, pathway plot and CO2 reduction tables
RUN_NEB    = False   # NEB energy table, barriers and band plots
RUN_MD     = False   # MD trajectory analysis (site occupancy, desorption)
RUN_PAPER  = False   # Paper figures and tables (run_analysis paper)

RUN_DIR = LUCAS_DIR / "results" / "simulations" / RUN_ID
ANALYSES_DIR = LUCAS_DIR / "results" / "analyses"
POST_ENV = {**os.environ, "QML_ZNO_RESULTS_DIR": str(RUN_DIR)}

def run_post(module: str, *args: str) -> None:
    """Read-only module on this run (it reads QML_ZNO_RESULTS_DIR)."""
    run_cmd([sys.executable, "-m", module, *args], env=POST_ENV)

def run_analysis(method: str, section: str) -> Path:
    """The launcher's command for one method or alias; returns the new analysis run."""
    analysis_id = f"{OUTPUT_ID}_{section}"
    run_cmd([
        sys.executable, "-m", "scripts.workflows.run_analysis", method,
        "--config", str(LUCAS_DIR / ANALYSIS_CONFIG),
        "--scientific-results", str(RUN_DIR),
        "--run-id", analysis_id,
    ])
    return ANALYSES_DIR / analysis_id

print(f"Run directory: {RUN_DIR}" + ("" if RUN_DIR.is_dir() else "  [missing]"))
print(f"Analysis runs: {ANALYSES_DIR / OUTPUT_ID}_<section>")

## 1) Run status

Read-only dashboard of the run (methods, sites, progress, stale calculations).
Same as `python -m scripts.workflows.run_status <RUN_ID>`; it writes nothing.

In [ ]:
if RUN_STATUS:
    run_post("scripts.workflows.run_status", RUN_ID, "--no-color")

## 2) CO2RR pathway and CO2 reduction tables

`run_analysis co2_reduction` (its dependency `co2rr_corrections` is added
automatically): CHE free energies in variants a-c with the pathway plot of the
configured variant (`science/co2rr/`), then the merged DFT/TDDFT/CO2RR tables
(`science/co2_reduction/`).

Sites whose intermediates desorb or change identity get no limiting potential. The site
ordering is not resolved when differences are within ~0.05 eV in E_ads or ~0.1 V in U_L.

In [ ]:
if RUN_CO2RR:
    co2rr_dir = run_analysis("co2_reduction", "co2rr")
    show_table(co2rr_dir / "science" / "co2rr" / "co2rr_corrected_summary.csv",
               ["site", "variant", "status", "pathway_validity", "dG_CO2_to_COOH_ev", "dG_COOH_to_CO_ev",
                "dG_CO_desorption_ev", "limiting_potential_v"],
               material=MATERIAL)
    list_files(co2rr_dir / "science")

## 3) NEB

`run_analysis neb`: `neb_energy_table` recomputes the barriers from the stored band
energies (`neb_final_images.traj`) and `plot_neb` draws each band and the barrier
comparison, into `science/neb/`.

Barriers below the resolution (0.03 eV), and bands whose endpoints are not minima along
the band, are reported without a barrier.

In [ ]:
if RUN_NEB:
    neb_dir = run_analysis("neb", "neb") / "science" / "neb"
    show_table(neb_dir / "neb_barriers.csv")
    list_files(neb_dir, patterns=("*.png", "*.csv"))

## 4) Molecular dynamics — trajectory analysis

`run_analysis md_trajectory`: per trajectory (site, seed), time adsorbed at the starting
site, lateral displacement and desorption (persistent CO2-based metric; the sensitivity
table repeats it for 4.0, 4.5 and 5.0 A). It runs on a partial MD run; partial,
unreadable and overfull trajectories are flagged in `status`. Tables and plot go to
`science/md/`; only `md_frame_analysis.csv` is written into the simulation run.

In [ ]:
if RUN_MD:
    md_dir = run_analysis("md_trajectory", "md") / "science" / "md"
    show_table(md_dir / "md_trajectory_analysis_summary.csv",
               ["site", "seed", "status", "production_time_fs", "fraction_production_adsorbed_at_start_site",
                "fraction_production_desorbed_persistent", "final_lateral_displacement_ang",
                "mean_c_o_min_slab_distance_ang"])
    show_table(md_dir / "md_desorption_sensitivity.csv")
    list_files(md_dir, patterns=("*.png",))

## 5) Paper figures and tables

`run_analysis paper`: CO2RR corrections, NEB, MD and structures analyses, then fig01
(two variants: C–O lengths and shortest CO2–surface contact), fig03-fig07 (each with a
`_data.csv`) and table02 in `paper/`, with `figure_registry.csv` (item, file, producer,
inputs, sha256). Items and caveats: [docs/figures.md](../docs/figures.md).

In [ ]:
if RUN_PAPER:
    paper_dir = run_analysis("paper", "paper") / "paper"
    show_table(paper_dir / "figure_registry.csv", ["item", "file", "producer_module", "note"])
    list_files(paper_dir, patterns=("*.png", "*.pdf", "*.csv", "*.md", "*.tex"))